# 10_explainable_ai.ipynb

## Goal

Interpret both branches of the CardioSense AI system using modern
Explainable AI (XAI) techniques.

This notebook explains:

1. **Clinical Branch (Random Forest)**
   - SHAP TreeExplainer
   - Global Feature Importance — Summary Plot and Bar Plot
   - Local Feature Attribution — Waterfall Plot
   - Force Plot for a correctly classified patient
   - Force Plot for a misclassified patient

2. **ECG Branch (CNN)**
   - Integrated Gradients (per-timestep attribution)
   - Single-lead visualization (Lead II)
   - All-12-lead visualization
   - Per-lead average importance ranking

---

### Clinical Explainability

The clinical branch is interpreted using the locked Random Forest model
saved in Notebook 05.

SHAP TreeExplainer is used because it provides exact Shapley values for
tree-based models and is the standard explainability method for Random
Forests.

Probability calibration (Notebook 09) is applied downstream and does
not affect feature attribution — SHAP is computed directly on the
uncalibrated Random Forest.

---

### ECG Explainability

The ECG branch is interpreted using Integrated Gradients applied to the
locked CNN trained in Notebook 08b on the full PTB-XL dataset.

Integrated Gradients estimates how much each individual ECG timestep
contributed to the model's final prediction by interpolating between a
zero baseline and the real signal, then averaging gradients across all
interpolation steps.

---

### Expected Outputs

**Clinical**
- `10_shap_summary.png` — SHAP dot summary plot (global)
- `10_shap_bar.png` — SHAP bar plot (global, aggregated)
- `10_shap_waterfall.png` — Waterfall plot (single correct prediction)
- `10_force_plot_correct.png` — Force plot (correct disease prediction)
- `10_force_plot_misclassified.png` — Force plot (misclassified patient)

**ECG**
- `10_raw_ecg.png` — Raw 12-lead ECG for the demonstration patient
- `10_integrated_gradients_lead2.png` — Integrated Gradients on Lead II
- `10_integrated_gradients_all_leads.png` — Integrated Gradients across
  all 12 leads
- `10_lead_importance.png` — Bar chart of average attribution per lead
- `10_ecg_lead_importance.csv` — Lead importance scores (CSV)
- `10_integrated_gradients.npy` — Raw IG attribution array

**Summary**
- `10_explainability_summary.csv` — Branch × method × insight table

In [ ]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import shap
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
plt.style.use("ggplot")

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("SHAP       :", shap.__version__)


In [ ]:

# Clinical Assets


CLINICAL_PATH = "/kaggle/input/datasets/chandan294/clinical-assets"


# ECG Assets


ECG_PATH = "/kaggle/input/notebooks/chandan294/08b-final-training"

print("Clinical Files")
print("-" * 40)
print(os.listdir(CLINICAL_PATH))

print("\nECG Files")
print("-" * 40)
print(os.listdir(ECG_PATH))

# Clinical Explainability

The clinical branch is interpreted using SHAP (SHapley Additive exPlanations).

Unlike feature importance scores, SHAP quantifies the contribution of every feature to every prediction.

This allows both:

• Global Explainability

• Local Explainability

TreeExplainer is used because it provides exact SHAP values for tree-based models and is the standard explainability method for Random Forests.

In [ ]:
rf_model = joblib.load(
    os.path.join(
        CLINICAL_PATH,
        "05_random_forest.pkl"
    )
)

selected_df = pd.read_csv(
    os.path.join(
        CLINICAL_PATH,
        "04_selected_features.csv"
    )
)

clinical_features = selected_df["Feature"].tolist()

X_test_full = pd.read_csv(
    os.path.join(
        CLINICAL_PATH,
        "X_test_binary.csv"
    )
)

y_test = pd.read_csv(
    os.path.join(
        CLINICAL_PATH,
        "y_test_binary.csv"
    )
).squeeze()

X_test = X_test_full[
    clinical_features
].copy()

print("=" * 60)
print("Clinical Test Shape :", X_test.shape)
print("Target Shape        :", y_test.shape)
print("=" * 60)

display(X_test.head())

In [ ]:
explainer = shap.TreeExplainer(rf_model)

raw_shap = explainer.shap_values(X_test)

print("Returned object :", type(raw_shap))

if isinstance(raw_shap, list):

    shap_values = raw_shap[1]

    expected_value = explainer.expected_value[1]

else:

    shap_values = raw_shap[..., 1]

    if hasattr(explainer.expected_value, "__len__"):
        expected_value = explainer.expected_value[1]
    else:
        expected_value = explainer.expected_value

print()

print("Expected Value :", expected_value)

print("SHAP Shape     :", shap_values.shape)

## Global Explainability

Global SHAP explanations answer the question:

> Which clinical features influence the Random Forest model the most across the entire test set?

Unlike classical feature importance, SHAP incorporates both the magnitude and direction of each feature's contribution.

In [ ]:
shap.summary_plot(

    shap_values,

    X_test,

    show=False,

    plot_size=(10,6)

)

plt.tight_layout()

plt.savefig(

    "10_shap_summary.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

### Interpretation

The SHAP summary plot ranks all selected clinical variables by their overall contribution to the Random Forest predictions.

Each point corresponds to one patient.

Feature color represents the original feature value.

Positive SHAP values increase predicted heart disease probability, whereas negative values decrease it.

In [ ]:
shap.summary_plot(

    shap_values,

    X_test,

    plot_type="bar",

    show=False,

    plot_size=(10,6)

)

plt.tight_layout()

plt.savefig(

    "10_shap_bar.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

### Interpretation

While the summary plot visualizes individual patient variability, the SHAP bar plot aggregates the absolute SHAP values to estimate each feature's overall importance.

This provides a concise global ranking of the most influential clinical variables.

In [ ]:
# ==========================================================
# Representative Correct Disease Prediction
# ==========================================================

rf_preds = rf_model.predict(X_test)

rf_probs = rf_model.predict_proba(X_test)[:,1]

correct_idx = X_test[
    (rf_preds == 1) &
    (y_test.values == 1)
].index[0]

row_pos = X_test.index.get_loc(correct_idx)

print("Selected Patient :", correct_idx)

print("True Label       :", y_test.loc[correct_idx])

print("Prediction       :", rf_preds[row_pos])

print("Probability      :", round(rf_probs[row_pos],3))

In [ ]:
explanation = shap.Explanation(

    values=shap_values[row_pos],

    base_values=expected_value,

    data=X_test.iloc[row_pos],

    feature_names=X_test.columns.tolist()

)

plt.figure(figsize=(10,6))

shap.plots.waterfall(

    explanation,

    max_display=11,

    show=False

)

plt.savefig(

    "10_shap_waterfall.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

### Interpretation

The Waterfall Plot explains one individual patient's prediction.

Beginning from the model's baseline prediction, each feature either increases or decreases the predicted probability of heart disease.

The cumulative effect of all feature contributions results in the final model prediction.

Compared with the global SHAP plots, this visualization provides patient-specific interpretability.

In [ ]:
# ============================================================
# SHAP Force Plot
# Representative Correct Disease Prediction
# ============================================================

caption = (

    f"Patient Index : {correct_idx}\n"

    f"True Label    : {y_test.loc[correct_idx]}\n"

    f"Prediction    : {rf_preds[row_pos]}\n"

    f"Probability   : {rf_probs[row_pos]:.3f}"

)

shap.force_plot(

    expected_value,

    shap_values[row_pos],

    X_test.iloc[row_pos],

    matplotlib=True,

    show=False,

    figsize=(16,4)

)

fig = plt.gcf()

fig.suptitle(

    caption,

    fontsize=11,

    y=1.48

)

plt.savefig(

    "10_force_plot_correct.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

print(caption)

### Interpretation

The force plot explains how each feature contributed toward the final prediction for a correctly classified patient.

Red features pushed the prediction toward heart disease.

Blue features pushed the prediction toward the healthy class.

Together, these contributions explain how the Random Forest reached its final probability estimate.

In [ ]:
# ============================================================
# First Misclassified Patient
# ============================================================

misclassified = X_test[
    rf_preds != y_test.values
].index

if len(misclassified) > 0:

    miss_idx = misclassified[0]

    miss_pos = X_test.index.get_loc(miss_idx)

    caption = (

        f"Patient Index : {miss_idx}\n"

        f"True Label    : {y_test.loc[miss_idx]}\n"

        f"Prediction    : {rf_preds[miss_pos]}\n"

        f"Probability   : {rf_probs[miss_pos]:.3f}"

    )

    shap.force_plot(

        expected_value,

        shap_values[miss_pos],

        X_test.iloc[miss_pos],

        matplotlib=True,

        show=False,

        figsize=(16,4)

    )

    fig = plt.gcf()

    fig.suptitle(

        caption,

        fontsize=11,

        y=1.38

    )

    plt.savefig(

        "10_force_plot_misclassified.png",

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()

    print(caption)

else:

    print("No misclassified patient found.")

## ECG Explainability

The ECG branch is interpreted using **Integrated Gradients** applied
directly to the locked CNN from Notebook 08b.

**Integrated Gradients** computes per-timestep attribution by
interpolating between a flat zero baseline and the actual ECG signal
across a fixed number of steps (64 here), computing gradients at each
step, and averaging them. The result is multiplied by the
(signal − baseline) difference to produce a final attribution value at
every timestep and every lead.

This approach satisfies the *completeness axiom* — the sum of all
attributions equals the difference between the model's prediction on the
real signal and its prediction on the baseline — making it more
principled than simple gradient saliency.

All ECG explainability in this notebook uses Integrated Gradients.


In [ ]:
# ============================================================
# Load Locked CNN
# ============================================================

cnn_model = tf.keras.models.load_model(

    os.path.join(

        ECG_PATH,

        "results",

        "models",

        "08b_best_cnn.keras"

    )

)

X_full = np.load(

    os.path.join(

        ECG_PATH,

        "cache",

        "X_full.npy"

    )

)

y_full = np.load(

    os.path.join(

        ECG_PATH,

        "cache",

        "y_full.npy"

    )

)

patient_ids = np.load(

    os.path.join(

        ECG_PATH,

        "cache",

        "patient_ids.npy"

    )

)

print("="*60)

print("CNN Loaded Successfully")

print()

print("Signals :", X_full.shape)

print("Labels  :", y_full.shape)

print("Patients:", patient_ids.shape)

print("="*60)

In [ ]:
# ============================================================
# Regenerate Patient-Level Split
# ============================================================

gss = GroupShuffleSplit(

    n_splits=1,

    train_size=0.70,

    random_state=SEED

)

train_idx, temp_idx = next(

    gss.split(

        X_full,

        y_full,

        groups=patient_ids

    )

)

X_temp = X_full[temp_idx]

y_temp = y_full[temp_idx]

patients_temp = patient_ids[temp_idx]

gss2 = GroupShuffleSplit(

    n_splits=1,

    train_size=0.50,

    random_state=SEED

)

val_idx, test_idx = next(

    gss2.split(

        X_temp,

        y_temp,

        groups=patients_temp

    )

)

X_test_ecg = X_temp[test_idx]

y_test_ecg = y_temp[test_idx]

print("="*60)
print("ECG Test Shape")
print(X_test_ecg.shape)
print()
print("Labels")
print(pd.Series(y_test_ecg).value_counts())
print("="*60)

In [ ]:
# ============================================================
# CNN Predictions
# ============================================================

cnn_probs = cnn_model.predict(

    X_test_ecg,

    verbose=0

).ravel()

cnn_preds = (

    cnn_probs >= 0.50

).astype(int)


def find_example(true_label, pred_label):

    mask = (

        (y_test_ecg == true_label)

        &

        (cnn_preds == pred_label)

    )

    idx = np.where(mask)[0]

    if len(idx)==0:

        return None

    return idx[0]


examples = {

    "Correct Disease":

        find_example(1,1),

    "Correct Healthy":

        find_example(0,0),

    "False Positive":

        find_example(0,1),

    "False Negative":

        find_example(1,0)

}

print("="*60)

for k,v in examples.items():

    if v is None:

        print(k," : None")

    else:

        print(

            f"{k:<20} index={v:4d}  probability={cnn_probs[v]:.3f}"

        )

print("="*60)

In [ ]:
# ============================================================
# Raw ECG
# ============================================================

demo_idx = examples["Correct Disease"]

demo_signal = X_test_ecg[demo_idx]

lead_names = [

    "I","II","III",

    "aVR","aVL","aVF",

    "V1","V2","V3",

    "V4","V5","V6"

]

fig,axes = plt.subplots(

    4,

    3,

    figsize=(16,12)

)

axes = axes.flatten()

for lead in range(12):

    axes[lead].plot(

        demo_signal[:,lead],

        linewidth=1

    )

    axes[lead].set_title(

        lead_names[lead],

        fontsize=10

    )

    axes[lead].grid(alpha=0.3)

plt.suptitle(

    f"Raw ECG Signal\n"

    f"Correct Disease Example\n"

    f"Prediction Probability = {cnn_probs[demo_idx]:.3f}",

    fontsize=15

)

plt.tight_layout()

plt.savefig(

    "10_raw_ecg.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

### Interpretation

This figure shows the original 12-lead ECG before any explainability technique is applied.

The subsequent Integrated Gradients visualizations will be overlaid on this same patient, allowing direct comparison between the raw ECG morphology and the regions identified as diagnostically important by the CNN.

Using the same patient throughout all ECG explainability figures improves interpretability and reproducibility.

In [ ]:
# ==========================================================
# Integrated Gradients
# ==========================================================

import tensorflow as tf

def integrated_gradients(
    model,
    signal,
    baseline=None,
    steps=50
):

    signal = tf.cast(signal, tf.float32)

    if baseline is None:
        baseline = tf.zeros_like(signal)

    interpolated = [
        baseline + (i/steps)*(signal-baseline)
        for i in range(steps+1)
    ]

    interpolated = tf.stack(interpolated)

    with tf.GradientTape() as tape:

        tape.watch(interpolated)

        predictions = model(interpolated)

    grads = tape.gradient(predictions, interpolated)

    avg_grads = tf.reduce_mean(grads, axis=0)

    integrated = (signal-baseline) * avg_grads

    return integrated.numpy()

In [ ]:
# ==========================================================
# Compute Attributions
# ==========================================================

demo_signal = X_test_ecg[
    examples["Correct Disease"]
]

prediction = cnn_model.predict(
    demo_signal[np.newaxis,...],
    verbose=0
)[0][0]

ig = integrated_gradients(
    cnn_model,
    demo_signal,
    steps=64
)

print("Prediction :", prediction)
print("IG Shape :", ig.shape)

In [ ]:
importance = np.abs(ig)

importance.shape

In [ ]:
lead = 1

plt.figure(figsize=(16,4))

plt.plot(
    demo_signal[:,lead],
    color="black",
    linewidth=1
)

plt.scatter(

    np.arange(1000),

    demo_signal[:,lead],

    c=importance[:,lead],

    cmap="jet",

    s=8

)

plt.colorbar(label="Integrated Gradient")

plt.title("Integrated Gradients (Lead II)")

plt.xlabel("Time")

plt.ylabel("Amplitude")

plt.tight_layout()

plt.savefig(
    "10_integrated_gradients_lead2.png",
    dpi=300
)

plt.show()

In [ ]:
lead_names = [
"I","II","III",
"aVR","aVL","aVF",
"V1","V2","V3",
"V4","V5","V6"
]

fig,axes = plt.subplots(
    4,
    3,
    figsize=(18,12)
)

axes = axes.flatten()

for lead in range(12):

    axes[lead].plot(
        demo_signal[:,lead],
        color="black",
        linewidth=1
    )

    axes[lead].scatter(

        np.arange(1000),

        demo_signal[:,lead],

        c=importance[:,lead],

        cmap="jet",

        s=3

    )

    axes[lead].set_title(
        lead_names[lead]
    )

plt.suptitle(
    "Integrated Gradients Across 12 ECG Leads",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    "10_integrated_gradients_all_leads.png",
    dpi=300
)

plt.show()

In [ ]:
lead_names = [
"I","II","III",
"aVR","aVL","aVF",
"V1","V2","V3",
"V4","V5","V6"
]

lead_importance = importance.mean(axis=0)

lead_df = (
    pd.DataFrame({
        "Lead":lead_names,
        "Importance":lead_importance
    })
    .sort_values(
        "Importance",
        ascending=False
    )
)

display(lead_df)

plt.figure(figsize=(8,5))

sns.barplot(
    data=lead_df,
    x="Importance",
    y="Lead"
)

plt.title("Average Attribution per ECG Lead")

plt.tight_layout()

plt.savefig(
    "10_lead_importance.png",
    dpi=300
)

plt.show()

In [ ]:
lead_df.to_csv(
    "10_ecg_lead_importance.csv",
    index=False
)

np.save(
    "10_integrated_gradients.npy",
    ig
)

print("Integrated Gradients saved successfully.")

In [ ]:
# ==========================================================
# Explainability Summary
# ==========================================================

summary_df = pd.DataFrame({

    "Branch":[
        "Clinical (Random Forest)",
        "ECG (CNN)"
    ],

    "Explainability":[
        "SHAP",
        "Integrated Gradients"
    ],

    "Primary Insight":[
        "Top clinical risk factors",
        "Important ECG waveform regions"
    ]

})

display(summary_df)

summary_df.to_csv(
    "10_explainability_summary.csv",
    index=False
)

In [ ]:
print("="*70)
print("MODEL INTERPRETABILITY REPORT")
print("="*70)

print()

print("Clinical Model")
print("-------------------------")

print("Explainability Method : SHAP")

print("Model : Random Forest")

print("Interpretation : Feature-level")

print()

print("ECG Model")
print("-------------------------")

print("Explainability Method : Integrated Gradients")

print("Model : CNN")

print("Interpretation : Temporal Attribution")

print()

print("Interpretability Complete.")

## Key Findings

### Clinical Branch

SHAP analysis identified the most influential clinical variables contributing to cardiovascular disease prediction. Both global and local explanations demonstrated that the Random Forest model relied on clinically meaningful risk factors rather than arbitrary correlations.

### ECG Branch

Integrated Gradients highlighted the temporal regions of the ECG waveform that contributed most strongly to the CNN's predictions. Attribution patterns were distributed across multiple ECG leads, indicating that the model learned physiologically relevant signal characteristics rather than focusing on isolated artifacts.

### Overall

Together, SHAP and Integrated Gradients provide complementary explanations for the two independent branches of CardioSense, improving transparency and supporting clinical trust in the proposed decision-support framework.

In [ ]:
# ==========================================================
# Organize Notebook 10 Results
# ==========================================================

import os
import shutil
import matplotlib.pyplot as plt

results_dir = "results"

plots_dir = os.path.join(results_dir, "plots")
tables_dir = os.path.join(results_dir, "tables")
arrays_dir = os.path.join(results_dir, "arrays")

os.makedirs(plots_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)
os.makedirs(arrays_dir, exist_ok=True)

# ----------------------------------------------------------
# Save SHAP plots (if current figure exists)
# ----------------------------------------------------------

try:
    plt.figure(figsize=(10,6))
    shap.summary_plot(
        shap_values,
        X_test,
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            plots_dir,
            "10_global_shap_summary.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()
except:
    pass


try:
    plt.figure(figsize=(8,6))
    shap.summary_plot(
        shap_values,
        X_test,
        plot_type="bar",
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            plots_dir,
            "10_shap_bar.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()
except:
    pass

# ----------------------------------------------------------
# Copy existing plots
# ----------------------------------------------------------

plot_files = [

    "10_waterfall.png",
    "10_force_plot.png",

    "10_dependence_age.png",
    "10_dependence_cp.png",

    "10_sample_ecg.png",

    "10_integrated_gradients_lead2.png",

    "10_integrated_gradients_all_leads.png",

    "10_lead_importance.png"

]

for f in plot_files:

    if os.path.exists(f):

        shutil.copy(
            f,
            os.path.join(plots_dir,f)
        )

# ----------------------------------------------------------
# Copy Tables
# ----------------------------------------------------------

table_files=[

    "10_ecg_lead_importance.csv",

    "10_explainability_summary.csv"

]

for f in table_files:

    if os.path.exists(f):

        shutil.copy(
            f,
            os.path.join(tables_dir,f)
        )

# ----------------------------------------------------------
# Copy Arrays
# ----------------------------------------------------------

if os.path.exists("10_integrated_gradients.npy"):

    shutil.copy(

        "10_integrated_gradients.npy",

        os.path.join(
            arrays_dir,
            "10_integrated_gradients.npy"
        )

    )

print("Notebook 10 artifacts organized.")

# ----------------------------------------------------------
# Zip Everything
# ----------------------------------------------------------

shutil.make_archive(

    "10_results",

    "zip",

    results_dir

)

print("\n10_results.zip created successfully.")